In [4]:
import pandas as pd

In [5]:
movies = pd.read_csv('../data/movies.csv/movies.csv')

actors = pd.read_csv('../data/actors.csv/actors.csv')
#only directors
directors = pd.read_csv('../data/crew.csv/crew.csv').query('role == "Director"')
genre = pd.read_csv('../data/genres.csv/genres.csv')
themes = pd.read_csv('../data/themes.csv/themes.csv')
posters = pd.read_csv('../data/posters.csv/posters.csv')


In [6]:
movies_clean = movies.dropna(subset=['name','description'])
movies_clean['tagline']= movies_clean['tagline'].fillna('')

C:\Users\catas\AppData\Local\Temp\ipykernel_23708\2923793274.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_clean['tagline']= movies_clean['tagline'].fillna('')


In [7]:
#join movies with actors
top_actors = actors.groupby('id').head(3)
actors_grouped = top_actors.dropna(subset=['name']).groupby('id')['name'].apply(', '.join).reset_index()
actors_grouped.rename(columns={'name': 'main_actors'}, inplace=True)


In [8]:
directors_grouped = directors.dropna(subset=['name']).groupby('id')['name'].apply(', '.join).reset_index()
directors_grouped.rename(columns={'name': 'main_directors'}, inplace=True)


In [9]:
genres_grouped = genre.dropna(subset=['genre']).groupby('id')['genre'].apply(', '.join).reset_index()
genres_grouped.rename(columns={'genre': 'main_genres'}, inplace=True)


In [10]:
themes_grouped = themes.dropna(subset=['theme']).groupby('id')['theme'].apply(', '.join).reset_index()
themes_grouped.rename(columns={'theme': 'main_themes'}, inplace=True)


In [11]:
movies_enriched = movies_clean.merge(
    actors_grouped, on='id', how='left') \
    .merge(directors_grouped, on='id', how='left') \
    .merge(genres_grouped, on='id', how='left') \
    .merge(themes_grouped, on='id', how='left')


movies_enriched.head()
movies_enriched.count()


id                780783
name              780783
date              713354
tagline           780783
description       780783
minute            697839
rating             90403
main_actors       538675
main_directors    677012
main_genres       574072
main_themes        24498
dtype: int64

In [13]:

movies_enriched = movies_enriched.merge(posters[['id', 'link']], on='id', how='left')

In [16]:
columns_to_fill = ['main_actors', 'main_directors', 'main_genres', 'main_themes', 'link']
movies_enriched[columns_to_fill] = movies_enriched[columns_to_fill].fillna('')

In [17]:
movies_enriched.count()


id                780783
name              780783
date              713354
tagline           780783
description       780783
minute            697839
rating             90403
main_actors       780783
main_directors    780783
main_genres       780783
main_themes       780783
link              780783
dtype: int64

In [18]:
#save enriched data
movies_enriched.to_csv('../data/movies_enriched.csv', index=False)